# 09b — Calibration of a new user: head-to-head with published identity-shrunk Procrustes

All methods share the same preprocessing (moment matching with the statistics of the k calibration trials, shrunk to the source-subject statistics, n0 = 50), the same calibration subsets (nested: the k-subset is a prefix of one permutation per draw), the same frozen route_c LOSO model and the same CSLS scoring. Only the map T = (1 − α)I + α·R_ρ differs.

| Method | α | ρ | Source |
|---|---|---|---|
| `identity` | 0 | – | moment matching only |
| `Procrustes` | 1 | 0 | Schönemann 1966 (B1) |
| `Procrustes rho=0.1` | 1 | 0.1 | 08 baseline |
| `beta-Procrustes CV` | 3-fold CV on the k pairs, grid 0…1 step 0.05 | 0 | **Ouali et al., ICCV 2023**, Eq. 5 (B2) |
| `fixed alpha=0.25` | 0.25 | 0 | **Cao et al., Signal Processing 2027**, Eq. 6; = Ouali β = 0.75 (B3) |
| `SHRINK` | k/(k+100) | 0.1(1+100/k) | ours (09) |
| `adaptive alpha only` | k/(k+100) | 0.1 | ablation |
| `adaptive rho only` | 1 | 0.1(1+100/k) | ablation |
| `closed-form OOF alpha (D)` | clip(Σaᵢᵀbᵢ / Σ‖aᵢ‖²), out-of-fold (LOO for k < 20, 5-fold otherwise); ρ ∈ {0, 0.1, 1} by out-of-fold error | | proposed estimator (not found published) |
| `D + pseudo (transductive)` | D, with 64 test-time pseudo-landmarks added to the pairs | | transductive |

k = 0, 1, 5, 10, 20, 50, 100, 200, 500, 1000; 3 draws; 10 LOSO folds. Metrics: Top-1, negative-transfer rate (subject × draw below `identity` at the same k, and below k = 0), worst subject, area under Top-1 vs log(1+k).
**Drive is full:** reads only; results zipped and downloaded at the end. **GPU:** L4 (~40–50 min incl. ~10 min retraining of missing models).

In [ ]:
# 0. Setup
import os, sys, gc, json, math, time, random, shutil, glob, copy, importlib.util, subprocess
try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    pass
DRIVE  = os.environ.get("EEG_DRIVE", "/content/drive/MyDrive/eeg_results")
DCACHE = f"{DRIVE}/cache"
ROOT   = os.environ.get("EEG_ROOT", "/content/EEG_Data")
LOCAL  = os.environ.get("EEG_LOCAL", "/content/cache")
OUT    = "/content/v9b_local"   # runtime disk (Drive is full); zipped and downloaded at the end
V6, V6B = f"{DRIVE}/v6", f"{DRIVE}/v6b"
for d in [DRIVE, DCACHE, ROOT, LOCAL, OUT]: os.makedirs(d, exist_ok=True)
ALL_SUBJECTS = [f"sub-{i:02d}" for i in range(1, 11)]
SMOKE = os.environ.get("EEG_SMOKE") == "1"           # tiny synthetic run for code checking only

import numpy as np
for pkg, pip_name in [("huggingface_hub", "huggingface_hub"), ("pandas", "pandas"), ("scipy", "scipy")]:
    if not SMOKE and importlib.util.find_spec(pkg) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip_name], check=True)
import pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
device = "cuda" if torch.cuda.is_available() else "cpu"
AMP = device == "cuda" and torch.cuda.get_device_capability()[0] >= 8      # bfloat16 needs Ampere or newer (not T4)
print("device:", device, torch.cuda.get_device_name(0) if device == "cuda" else "", "| bf16:", AMP, "| torch", torch.__version__)
print(f"Drive free: {shutil.disk_usage(DRIVE).free / 1e9:.1f} GB (features need ~1.2 GB)")

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)

def local_copy(name):
    dst = f"{LOCAL}/{name}"
    if not os.path.exists(dst): shutil.copy(f"{DCACHE}/{name}", dst + ".tmp"); os.replace(dst + ".tmp", dst)
    return dst

def save_json(obj, path):
    json.dump(obj, open(path + ".tmp", "w")); os.replace(path + ".tmp", path)

def free_gpu():
    gc.collect()
    if device == "cuda": torch.cuda.empty_cache()

In [ ]:
# 1. Image list (same order as notebooks 04/05) and the validation split
META = f"{DCACHE}/meta_v6.pt"
if not os.path.exists(META):
    import tarfile
    from huggingface_hub import hf_hub_download
    if not os.path.exists(f"{ROOT}/images_set/image_metadata.npy"):
        tar_path = hf_hub_download(repo_id="LidongYang/EEG_Image_decode", filename="images_set.tar.gz", repo_type="dataset", local_dir=ROOT)
        with tarfile.open(tar_path, "r:gz") as t: t.extractall(path=ROOT)
    obj = np.load(f"{ROOT}/images_set/image_metadata.npy", allow_pickle=True)
    meta = obj.item() if isinstance(obj, np.ndarray) and obj.dtype == object and obj.ndim == 0 else obj
    train_files, test_files = list(meta["train_img_files"]), list(meta["test_img_files"])
    def build_index(folder):
        idx = {}
        for dp, _, fs in os.walk(folder):
            for f in fs: idx.setdefault(f, os.path.join(dp, f))
        return idx
    img_root = f"{ROOT}/images_set"
    idx_train = build_index(f"{img_root}/training_images")
    idx_test = build_index(f"{img_root}/test_images") if os.path.isdir(f"{img_root}/test_images") else build_index(img_root)
    torch.save({"train_files": train_files, "test_files": test_files,
                "train_rel": [os.path.relpath(idx_train[f], ROOT) for f in train_files],
                "test_rel": [os.path.relpath(idx_test[f], ROOT) for f in test_files],
                "n_concepts": len(set(meta["train_img_concepts"]))}, META)
M = torch.load(META, weights_only=False)
N_TR, N_TE = len(M["train_files"]), len(M["test_files"])
PER = N_TR // M["n_concepts"]
IS_VAL = (np.arange(N_TR) % PER) == (PER - 1)              # last image of every training concept (same as 01–05)
print(f"train images {N_TR} | test images {N_TE} | concepts {M['n_concepts']} | val images {IS_VAL.sum()}")

In [ ]:
# 2. Cached image features of both backbones (written by notebooks 06 / 06b) and z-scored targets (training statistics)
def load_feats(b):
    p = f"{DCACHE}/feat6_{b}.pt"
    if os.path.exists(p): return torch.load(p, weights_only=False)
    parts = f"{DCACHE}/feat6_{b}_parts"                     # 06b saved chunks; the merged file may not have synced
    assert os.path.isdir(parts), f"neither {p} nor {parts} found"
    def merge(tag):
        files = sorted(glob.glob(f"{parts}/{tag}_*.pt")); print(f"  {b} {tag}: {len(files)} chunks")
        ch = [torch.load(f) for f in files]
        return {k: torch.cat([c[k] for c in ch]) for k in ch[0]}
    tr, te = merge("train"), merge("test")
    layers = sorted({l for l, _ in tr})
    assert len(tr[(layers[0], "mean")]) == N_TR and len(te[(layers[0], "mean")]) == N_TE, "chunks incomplete"
    return {"layers": layers, "tr": tr, "te": te}

FE = {b: load_feats(b) for b in ["internvit6b", "dinov2g"]}
for b, d in FE.items(): print(b, "layers", d["layers"], "dim", d["tr"][(d["layers"][0], "mean")].shape[1])

def target_te(b, layer):
    tr, te = FE[b]["tr"][(layer, "mean")].float(), FE[b]["te"][(layer, "mean")].float()
    m, s = tr.mean(0, keepdim=True), tr.std(0, keepdim=True) + 1e-4
    return ((te - m) / s).to(device)

CH63 = ["Fp1","Fp2","AF7","AF3","AFz","AF4","AF8","F7","F5","F3","F1","F2","F4","F6","F8","FT9","FT7","FC5","FC3","FC1",
        "FCz","FC2","FC4","FC6","FT8","FT10","T7","C5","C3","C1","Cz","C2","C4","C6","T8","TP9","TP7","CP5","CP3","CP1",
        "CPz","CP2","CP4","CP6","TP8","TP10","P7","P5","P3","P1","Pz","P2","P4","P6","P8","PO7","PO3","POz","PO4","PO8",
        "O1","Oz","O2"]
_, N_CH_ALL, N_T = np.load(f"{DCACHE}/sub-01_test_z16.npy", mmap_mode="r").shape
def load_test(sub):
    x = np.load(local_copy(f"{sub}_test_z16.npy"), mmap_mode="r")
    return torch.from_numpy(np.array(x)).to(device)
def load_train(sub):
    return torch.from_numpy(np.array(np.load(local_copy(f"{sub}_train_z16.npy"), mmap_mode="r"))).to(device)
def target_tr(b, layer):
    tr = FE[b]["tr"][(layer, "mean")].float()
    m, s = tr.mean(0, keepdim=True), tr.std(0, keepdim=True) + 1e-4
    return ((tr - m) / s).to(device)
VAL = torch.from_numpy(np.where(IS_VAL)[0]).to(device)


In [ ]:
# 4. Model: EEGProject (MLP with residual bottleneck, as in Shallow Alignment) + linear image projector(s)
#    Single-layer target, or a learned softmax routing over several layers (global SAMGA-style prior, no subject residual).
EMB = 1024

class EEGProject(nn.Module):
    def __init__(self, n_in, d=EMB, drop=0.3):
        super().__init__()
        self.inp = nn.Linear(n_in, d)
        self.res = nn.Sequential(nn.GELU(), nn.Linear(d, d), nn.Dropout(drop))
        self.norm = nn.LayerNorm(d)
    def forward(self, x):
        h = self.inp(x.flatten(1)); return self.norm(h + self.res(h))

class ImageSide(nn.Module):
    def __init__(self, dims, init_logits=None):
        super().__init__()
        self.proj = nn.ModuleList([nn.Linear(d, EMB) for d in dims])
        self.logits = nn.Parameter(torch.tensor(init_logits, dtype=torch.float) if init_logits is not None else torch.zeros(len(dims)),
                                   requires_grad=len(dims) > 1)
    def weights(self): return F.softmax(self.logits, 0)
    def forward(self, feats):                   # feats: list of (N, d_l), same rows
        w = self.weights()
        return sum(w[i] * p(f) for i, (p, f) in enumerate(zip(self.proj, feats)))

class Aligner(nn.Module):
    def __init__(self, n_in, dims, init_logits=None):
        super().__init__()
        self.eeg, self.img = EEGProject(n_in), ImageSide(dims, init_logits)
        self.log_scale = nn.Parameter(torch.tensor(math.log(1 / 0.07)))
    def scale(self): return self.log_scale.exp().clamp(max=100)

def clip_loss(ze, zi, scale):
    ze, zi = F.normalize(ze, dim=-1), F.normalize(zi, dim=-1)
    logits = scale * ze @ zi.T; y = torch.arange(len(ze), device=ze.device)
    return (F.cross_entropy(logits, y) + F.cross_entropy(logits.T, y)) / 2

@torch.no_grad()
def embed_eeg(model, X, bs=4096):
    model.eval(); out = []
    for i in range(0, len(X), bs):
        with torch.autocast(device_type="cuda", dtype=torch.bfloat16, enabled=AMP):
            out.append(F.normalize(model.eeg(X[i:i + bs].float()).float(), dim=-1))
    return torch.cat(out)

@torch.no_grad()
def embed_img(model, feats):
    model.eval(); return F.normalize(model.img(feats).float(), dim=-1)

@torch.no_grad()
def metrics(sim):
    # sim: queries x candidates, the correct candidate of query i is candidate i
    r = (sim > sim.diag().unsqueeze(1)).sum(1)
    return {"top1": (r < 1).float().mean().item() * 100, "top5": (r < 5).float().mean().item() * 100}

EPOCHS, BATCH, LR, WD, EVAL_EVERY = (60, 1024, 1e-4, 1e-4, 5) if not SMOKE else (3, 64, 1e-3, 1e-4, 1)
ROUTE_INIT = [-2.0, -1.0, 0.0, -1.0, -2.0]

def train(X, img_of, T_tr, val_sets, seed=0, log=None, init_logits=None):
    # X: EEG trials, img_of[i]: training image of trial i, T_tr: list of target matrices (N_TR x d_l)
    # val_sets: list of (EEG rows, image rows) used for model selection (held-out training images)
    set_seed(seed)
    model = Aligner(X[0].numel(), [t.shape[1] for t in T_tr], init_logits).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)
    tr_rows = torch.where(~torch.from_numpy(IS_VAL).to(device)[img_of])[0]
    steps = EPOCHS * math.ceil(len(tr_rows) / BATCH)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=LR, total_steps=steps, pct_start=0.05)
    best, best_state, best_ep = -1, None, 0
    for ep in range(1, EPOCHS + 1):
        model.train(); perm = tr_rows[torch.randperm(len(tr_rows), device=device)]
        for i in range(0, len(perm), BATCH):
            b = perm[i:i + BATCH]
            if len(b) < 2: continue
            imgs = img_of[b]
            with torch.autocast(device_type="cuda", dtype=torch.bfloat16, enabled=AMP):
                ze = model.eeg(X[b].float()); zi = model.img([t[imgs] for t in T_tr])
            loss = clip_loss(ze.float(), zi.float(), model.scale())
            opt.zero_grad(set_to_none=True); loss.backward(); opt.step(); sched.step()
        if ep % EVAL_EVERY == 0 or ep == EPOCHS:
            zi_val = embed_img(model, [t[VAL] for t in T_tr])
            va = float(np.mean([metrics(embed_eeg(model, X[er]) @ zi_val[ir].T)["top1"] for er, ir in val_sets]))
            if log: log(ep, va, model)
            if va > best: best, best_ep, best_state = va, ep, copy.deepcopy(model.state_dict())
    model.load_state_dict(best_state)
    return model, best_ep, best
@torch.no_grad()
def metrics(sim):
    r = (sim > sim.diag().unsqueeze(1)).sum(1)
    return {"top1": (r < 1).float().mean().item() * 100, "top5": (r < 5).float().mean().item() * 100}

AMP = AMP if "AMP" in globals() else False
# the four LOSO models of every fold: (name, folder, file tag, backbone, layers)
MODELS = [("intern_route", V6B, "route_c", "internvit6b", [20, 24, 28, 32, 36]),
          ("intern_L28",   V6B, "L28",     "internvit6b", [28]),
          ("dino_route",   V6,  "route",   "dinov2g",     [20, 24, 28, 32, 36]),
          ("dino_L20",     V6,  "L20",     "dinov2g",     [20])]
PRIMARY = "intern_route"

MODEL_LOCAL = f"{LOCAL}/models"; os.makedirs(MODEL_LOCAL, exist_ok=True)
ROUTE_INIT5 = [-2.0, -1.0, 0.0, -1.0, -2.0]

def model_path(folder, tag, sub):
    loc = f"{MODEL_LOCAL}/{os.path.basename(folder)}_loso_{tag}_{sub}.pt"
    return loc if os.path.exists(loc) else f"{folder}/loso_{tag}_{sub}.pt"

def retrain(folder, tag, sub, backbone, layers):
    """same recipe as notebooks 06/06b cell 6 (9 source subjects, 63 ch, seed 0, best epoch on source validation images)"""
    t0 = time.time(); srcs = [s for s in (ALL_SUBJECTS if not SMOKE else ALL_SUBJECTS[:3]) if s != sub]
    X = torch.empty((len(srcs) * N_TR, N_CH_ALL, N_T), dtype=torch.float16, device=device)
    for k, s_ in enumerate(srcs): X[k * N_TR:(k + 1) * N_TR] = load_train(s_)
    img_of = torch.arange(len(X), device=device) % N_TR
    vsets = [(k * N_TR + VAL, torch.arange(len(VAL), device=device)) for k in range(min(3, len(srcs)))]
    T_tr = [target_tr(backbone, l) for l in layers]
    m, ep, va = train(X, img_of, T_tr, vsets, 0, None, ROUTE_INIT5[-len(layers):] if len(layers) > 1 else None)
    del X, T_tr; free_gpu()
    loc = f"{MODEL_LOCAL}/{os.path.basename(folder)}_loso_{tag}_{sub}.pt"
    torch.save(m.state_dict(), loc)
    # Drive is full: keep the retrained model on the runtime disk only
    print(f"  retrained {tag} {sub}: best epoch {ep}, val {va:.1f} ({(time.time()-t0)/60:.1f} min)", flush=True)

def load_model(folder, tag, sub, backbone, layers):
    if not os.path.exists(model_path(folder, tag, sub)): retrain(folder, tag, sub, backbone, layers)
    dims = [FE[backbone]["tr"][(l, "mean")].shape[1] for l in layers]
    m = Aligner(N_CH_ALL * N_T, dims).to(device)
    m.load_state_dict(torch.load(model_path(folder, tag, sub), map_location=device)); m.eval()
    return m

missing = [(n, sub) for n, folder, tag, bb, layers in MODELS for sub in (ALL_SUBJECTS if not SMOKE else ALL_SUBJECTS[:2])
           if not os.path.exists(model_path(folder, tag, sub))]
print(f"{len(missing)} LOSO model files missing -> retrained in cell 4 (~1 min each on A100/L4, ~4 min on T4):", missing)


In [ ]:
# 3. Maps, scoring and embeddings
CSLS_K, RHO = 10, 0.1
from scipy.stats import wilcoxon

@torch.no_grad()
def csls(S, k=CSLS_K):
    rz = S.topk(min(k, S.shape[1]), dim=1).values.mean(1, keepdim=True)
    rt = S.topk(min(k, S.shape[0]), dim=0).values.mean(0, keepdim=True)
    return 2 * S - rz - rt

def mm(Z, ref_Z, ref_T):
    """moment matching: standardise Z with the statistics of ref_Z, rescale to the statistics of ref_T"""
    return F.normalize((Z - ref_Z.mean(0)) / (ref_Z.std(0) + 1e-6) * ref_T.std(0) + ref_T.mean(0), dim=-1)

def mnn_pairs(S):
    top2 = S.topk(2, dim=1); j = top2.indices[:, 0]; i = torch.arange(len(S), device=S.device)
    mnn = S.argmax(0)[j] == i
    margin = (top2.values[:, 0] - top2.values[:, 1]).clamp(min=1e-4)
    q = torch.where(mnn)[0]
    return q, j[q], margin[q]

def top_by(q, c, key, n):
    o = key.argsort(descending=True)[:n]; return q[o], c[o], key[o]

@torch.no_grad()
def fit_ortho(Xl, Yl, w=None, rho=RHO):
    w = torch.ones(len(Xl), device=Xl.device) if w is None else len(Xl) * w / w.sum()
    wn = w / w.sum(); muX = (wn[:, None] * Xl).sum(0); muY = (wn[:, None] * Yl).sum(0)
    C = ((Xl - muX) * w[:, None]).T @ (Yl - muY)
    Mx = C + rho * torch.linalg.matrix_norm(C, ord=2) * torch.eye(C.shape[0], device=C.device)
    U, _, Vh = torch.linalg.svd(Mx.double()); R = (U @ Vh).float()
    return lambda Z: F.normalize((Z - muX) @ R + muY, dim=-1)

@torch.no_grad()
def fit_ridge(Xl, Yl, lam_rel=1.0):
    """affine map pulled towards the identity: W = argmin |Xc W - Yc|^2 + lam |W - I|^2, lam = lam_rel * mean eigenvalue"""
    muX, muY = Xl.mean(0), Yl.mean(0); Xc, Yc = (Xl - muX).double(), (Yl - muY).double()
    d = Xc.shape[1]; G = Xc.T @ Xc; lam = lam_rel * torch.trace(G) / d + 1e-6
    I = torch.eye(d, device=Xl.device, dtype=torch.double)
    W = torch.linalg.solve(G + lam * I, Xc.T @ Yc + lam * I).float()
    return lambda Z: F.normalize((Z - muX) @ W + muY, dim=-1)

def rank_metrics(S):
    return metrics(S)

# z-scoring statistics of every (backbone, layer) on the training images, kept on CPU; targets built in chunks
STATS = {}
def stat(b, l):
    if (b, l) not in STATS:
        tr = FE[b]["tr"][(l, "mean")].float(); STATS[(b, l)] = (tr.mean(0, keepdim=True), tr.std(0, keepdim=True) + 1e-4)
    return STATS[(b, l)]

@torch.no_grad()
def embed_train_images(m, b, layers, rows=None, bs=2048):
    rows = torch.arange(N_TR) if rows is None else rows.cpu()
    out = []
    for i in range(0, len(rows), bs):
        r = rows[i:i + bs]
        feats = [((FE[b]["tr"][(l, "mean")][r].float() - stat(b, l)[0]) / stat(b, l)[1]).to(device) for l in layers]
        out.append(embed_img(m, feats))
    return torch.cat(out)


In [ ]:
# 4. Maps and calibration-only selection rules
N0, K0, RHO0 = 50, 100, 0.1
from scipy.stats import wilcoxon

def shrunk_stats(Zk, mu_src, sd_src, n0=N0):
    k = len(Zk)
    if k == 0: return mu_src, sd_src
    mu_k = Zk.mean(0); var_k = Zk.var(0, unbiased=False) if k > 1 else torch.zeros_like(mu_k)
    mu = (k * mu_k + n0 * mu_src) / (k + n0)
    var = (k * (var_k + (mu_k - mu) ** 2) + n0 * (sd_src ** 2 + (mu_src - mu) ** 2)) / (k + n0)
    return mu, var.sqrt()

def mm_stats(Z, mu, sd, T_ref):
    return F.normalize((Z - mu) / (sd + 1e-6) * T_ref.std(0) + T_ref.mean(0), dim=-1)

@torch.no_grad()
def ortho_R(X, Y, rho, w=None):
    """orthogonal Procrustes (row vectors, XR ~ Y) with optional pull to the identity; returns (R, muX, muY)"""
    w = torch.ones(len(X), device=X.device) if w is None else len(X) * w / w.sum()
    wn = w / w.sum(); muX = (wn[:, None] * X).sum(0); muY = (wn[:, None] * Y).sum(0)
    C = ((X - muX) * w[:, None]).T @ (Y - muY)
    M = C + rho * torch.linalg.matrix_norm(C, ord=2) * torch.eye(C.shape[0], device=C.device) if rho > 0 else C
    U, _, Vh = torch.linalg.svd(M.double()); return (U @ Vh).float(), muX, muY

def apply_map(Z, R, muX, muY, alpha):
    if alpha <= 0: return Z
    return F.normalize((1 - alpha) * Z + alpha * F.normalize((Z - muX) @ R + muY, dim=-1), dim=-1)

def folds_of(k, n_fold, g):
    p = torch.randperm(k, generator=g).to(device)
    return [p[i::n_fold] for i in range(n_fold)]

ALPHAS = torch.arange(0, 1.0001, 0.05).tolist()

def cv_alpha_retrieval(Xk, Yk, gallery_extra, g, rho=0.0, n_fold=3):
    """Ouali et al. 2023: choose alpha by 3-fold CV Top-1 on the calibration pairs (ties -> smaller alpha)"""
    k = len(Xk)
    if k < 3: return 0.0
    acc = torch.zeros(len(ALPHAS), device=device)
    for hold in folds_of(k, n_fold, g):
        fit = torch.ones(k, dtype=torch.bool, device=device); fit[hold] = False
        R, mx, my = ortho_R(Xk[fit], Yk[fit], rho)
        gal = torch.cat([Yk[hold], gallery_extra])           # held-out targets + distractor training images
        for a_i, a in enumerate(ALPHAS):
            S = apply_map(Xk[hold], R, mx, my, a) @ gal.T
            acc[a_i] += (S.argmax(1) == torch.arange(len(hold), device=device)).float().sum()
    best = acc.max(); return float(ALPHAS[int(torch.nonzero(acc >= best - 1e-9)[0])])

def oof_alpha(Xk, Yk, g, rhos=(0.0, 0.1, 1.0)):
    """estimator D: closed-form out-of-fold alpha minimising held-out squared error; rho by out-of-fold error"""
    k = len(Xk)
    if k < 2: return 0.0, 0.1
    folds = [torch.tensor([i], device=device) for i in range(k)] if k < 20 else folds_of(k, 5, g)
    best = None
    for rho in rhos:
        A, B = [], []
        for hold in folds:
            fit = torch.ones(k, dtype=torch.bool, device=device); fit[hold] = False
            if fit.sum() < 1: continue
            R, mx, my = ortho_R(Xk[fit], Yk[fit], rho)
            x = Xk[hold]; A.append(F.normalize((x - mx) @ R + my, dim=-1) - x); B.append(Yk[hold] - x)
        A, B = torch.cat(A), torch.cat(B)
        den = float((A * A).sum())
        a = 0.0 if den <= 1e-12 else min(1.0, max(0.0, float((A * B).sum()) / den))
        err = float(((B - a * A) ** 2).sum())
        if best is None or err < best[0]: best = (err, a, rho)
    return best[1], best[2]

def csls_k(S, k=10):
    rz = S.topk(min(k, S.shape[1]), dim=1).values.mean(1, keepdim=True)
    rt = S.topk(min(k, S.shape[0]), dim=0).values.mean(0, keepdim=True)
    return 2 * S - rz - rt


In [ ]:
# 5. Run: 10 LOSO folds x 10 budgets x 3 nested draws
FOLDS = ALL_SUBJECTS if not SMOKE else ALL_SUBJECTS[:2]
KS = [0, 1, 5, 10, 20, 50, 100, 200, 500, 1000] if not SMOKE else [0, 1, 5, 10]
DRAWS = 3 if not SMOKE else 1
N_SRC = 2000 if not SMOKE else 30
rows = []
name, folder, tag, bb, layers = MODELS[0]

for sub in FOLDS:
    t0 = time.time()
    m = load_model(folder, tag, sub, bb, layers)
    Xt, Xs = load_test(sub), load_train(sub)
    T = embed_img(m, [target_te(bb, l) for l in layers]); Ttr = embed_train_images(m, bb, layers)
    Zt, Zs = embed_eeg(m, Xt), embed_eeg(m, Xs)
    g0 = torch.Generator(device="cpu").manual_seed(0)
    Zsrc = torch.cat([embed_eeg(m, load_train(s)[torch.randperm(N_TR, generator=g0)[:N_SRC].to(device)])
                      for s in ALL_SUBJECTS if s != sub and (not SMOKE or s in ALL_SUBJECTS[:3])])
    mu_src, sd_src = Zsrc.mean(0), Zsrc.std(0); del Zsrc
    def add(method, k, d, S, alpha=float("nan"), rho=float("nan"), transductive=False):
        mt = metrics(S); rows.append(dict(subject=sub, method=method, k=k, draw=d, alpha=alpha, rho=rho,
                                          transductive=transductive, top1=mt["top1"], top5=mt["top5"]))
    add("direct (no calibration)", 0, 0, Zt @ T.T)
    for d in range(DRAWS):
        g = torch.Generator(device="cpu").manual_seed(100 + d)
        perm = torch.randperm(N_TR, generator=g).to(device)       # nested subsets: prefix of one permutation
        extra = Ttr[perm[-1000:]] if not SMOKE else Ttr[perm[-10:]]
        for k in KS:
            idx = perm[:k]; Zk, Yk = Zs[idx], Ttr[idx]
            mu, sd = shrunk_stats(Zk, mu_src, sd_src)
            Xk, Zq = mm_stats(Zk, mu, sd, Ttr), mm_stats(Zt, mu, sd, Ttr)
            sc = lambda Zm: csls_k(Zm @ T.T)
            add("identity", k, d, sc(Zq), 0.0)
            if k == 0:
                for meth in ["Procrustes", "Procrustes rho=0.1", "beta-Procrustes CV (Ouali 2023)", "fixed alpha=0.25 (Cao 2027)",
                             "SHRINK (ours)", "adaptive alpha only", "adaptive rho only", "closed-form OOF alpha (D)"]:
                    add(meth, 0, d, sc(Zq), 0.0)
                continue
            R0, mx0, my0 = ortho_R(Xk, Yk, 0.0); R1, mx1, my1 = ortho_R(Xk, Yk, RHO0)
            rk = RHO0 * (1 + K0 / k); ak = k / (k + K0); Rk, mxk, myk = ortho_R(Xk, Yk, rk)
            add("Procrustes", k, d, sc(apply_map(Zq, R0, mx0, my0, 1.0)), 1.0, 0.0)
            add("Procrustes rho=0.1", k, d, sc(apply_map(Zq, R1, mx1, my1, 1.0)), 1.0, RHO0)
            a_cv = cv_alpha_retrieval(Xk, Yk, extra, g)
            add("beta-Procrustes CV (Ouali 2023)", k, d, sc(apply_map(Zq, R0, mx0, my0, a_cv)), a_cv, 0.0)
            add("fixed alpha=0.25 (Cao 2027)", k, d, sc(apply_map(Zq, R0, mx0, my0, 0.25)), 0.25, 0.0)
            add("SHRINK (ours)", k, d, sc(apply_map(Zq, Rk, mxk, myk, ak)), ak, rk)
            add("adaptive alpha only", k, d, sc(apply_map(Zq, R1, mx1, my1, ak)), ak, RHO0)
            add("adaptive rho only", k, d, sc(apply_map(Zq, Rk, mxk, myk, 1.0)), 1.0, rk)
            a_d, r_d = oof_alpha(Xk, Yk, g); Rd, mxd, myd = ortho_R(Xk, Yk, r_d)
            add("closed-form OOF alpha (D)", k, d, sc(apply_map(Zq, Rd, mxd, myd, a_d)), a_d, r_d)
            # transductive: add 64 test-time pseudo-landmarks to the k labelled pairs, then estimator D
            S0 = sc(Zq); top2 = S0.topk(2, dim=1); j = top2.indices[:, 0]
            mnn = torch.where(S0.argmax(0)[j] == torch.arange(len(S0), device=device))[0]
            mg = (top2.values[mnn, 0] - top2.values[mnn, 1]); q = mnn[mg.argsort(descending=True)[:64]]
            Xa, Ya = torch.cat([Xk, Zq[q]]), torch.cat([Yk, T[j[q]]])
            a_p, r_p = oof_alpha(Xa, Ya, g); Rp, mxp, myp = ortho_R(Xa, Ya, r_p)
            add("D + pseudo (transductive)", k, d, sc(apply_map(Zq, Rp, mxp, myp, a_p)), a_p, r_p, True)
    R_ = pd.DataFrame([r for r in rows if r["subject"] == sub]).groupby(["method", "k"]).top1.mean()
    def gv(mn, k): return R_.get((mn, k), float("nan"))
    print(f"{sub} ({(time.time()-t0)/60:.1f} min) k=10: id {gv('identity',10):.1f} Proc {gv('Procrustes',10):.1f} Ouali {gv('beta-Procrustes CV (Ouali 2023)',10):.1f} "
          f"Cao {gv('fixed alpha=0.25 (Cao 2027)',10):.1f} SHRINK {gv('SHRINK (ours)',10):.1f} D {gv('closed-form OOF alpha (D)',10):.1f} | "
          f"k=200: Ouali {gv('beta-Procrustes CV (Ouali 2023)',200):.1f} SHRINK {gv('SHRINK (ours)',200):.1f} D {gv('closed-form OOF alpha (D)',200):.1f}", flush=True)
    del m, Xt, Xs, Ttr, Zs; free_gpu()

D9b = pd.DataFrame(rows); D9b.to_csv(f"{OUT}/calibration_v9b.csv", index=False)


In [ ]:
# 6. Summary
ORDER = ["identity", "Procrustes", "Procrustes rho=0.1", "beta-Procrustes CV (Ouali 2023)", "fixed alpha=0.25 (Cao 2027)",
         "SHRINK (ours)", "adaptive alpha only", "adaptive rho only", "closed-form OOF alpha (D)", "D + pseudo (transductive)"]
cur = D9b[D9b.method != "direct (no calibration)"].groupby(["method", "k"]).top1.mean().unstack("k").round(1)
print(f"direct (no calibration): {D9b[D9b.method == 'direct (no calibration)'].top1.mean():.1f}")
print("minutes per k:", {k: round(k * 4 * 0.2 / 60, 2) for k in KS})
print("\nTop-1 (mean over 10 subjects x 3 draws)")
print(cur.loc[[o for o in ORDER if o in cur.index]].to_string())

# negative transfer: per subject x draw, method < identity at the same k (isolates the map)
piv = D9b.pivot_table(index=["subject", "draw", "k"], columns="method", values="top1")
print("\nNegative-transfer rate vs identity at the same k (% of subject x draw cases worse than identity)")
nt = {}
for mth in ORDER[1:]:
    if mth in piv: nt[mth] = {k: round(float((piv.xs(k, level="k")[mth] < piv.xs(k, level="k")["identity"] - 1e-9).mean() * 100), 0) for k in KS if k > 0}
print(pd.DataFrame(nt).T.to_string())

# area under Top-1 vs log(1+k), and worst subject at k = 10, 100
lk = np.log1p(np.array(KS, dtype=float))
_trap = getattr(np, "trapezoid", None) or np.trapz
auc = {mth: float(_trap(np.nan_to_num(cur.loc[mth, KS].values.astype(float), nan=float(cur.loc["identity", 0])), lk) / (lk[-1] - lk[0])) for mth in ORDER if mth in cur.index}
ws = D9b.groupby(["method", "k", "subject"]).top1.mean()
print("\nmethod | area under Top-1 vs log(1+k) | worst subject k=10 | worst subject k=100 | mean alpha k=10 / k=1000")
for mth in ORDER:
    if mth not in cur.index: continue
    al = D9b[D9b.method == mth].groupby("k").alpha.mean()
    w10 = ws[(mth, 10)].min() if (mth, 10) in ws.index.droplevel(2) else float("nan")
    w100 = ws[(mth, 100)].min() if (mth, 100) in ws.index.droplevel(2) else float("nan")
    print(f"  {mth:34s} {auc[mth]:5.1f} | {w10:5.1f} | {w100:5.1f} | "
          f"{al.get(10, float('nan')):.2f} / {al.get(1000, float('nan')):.2f}")

print("\nOurs vs Ouali CV per k (mean diff, wins/10 subjects, Wilcoxon p)")
per = D9b.groupby(["method", "subject", "k"]).top1.mean()
for mine in ["SHRINK (ours)", "closed-form OOF alpha (D)"]:
    for k in [k for k in KS if k > 0]:
        a = per.xs((mine, k), level=("method", "k")); b = per.xs(("beta-Procrustes CV (Ouali 2023)", k), level=("method", "k"))
        dd = a - b; p = wilcoxon(dd).pvalue if (dd != 0).any() else float("nan")
        print(f"  {mine:28s} k={k:5d}: {dd.mean():+5.1f}  wins {int((dd > 0).sum())}/10  p={p:.3f}")

cur.to_csv(f"{OUT}/curve_v9b.csv")
import shutil
shutil.make_archive("/content/v9b_results", "zip", OUT)
try:
    from google.colab import files; files.download("/content/v9b_results.zip")
except Exception as e: print("download skipped:", e)
